# 05 · M1: Beta-Binomial jerárquico (Gibbs + Metropolis-Hastings)
**Grupo 1 · Estadística Bayesiana · Entrega final**  
Jean Paul Rojas Rojas · Daniel Estiven Pacheco Villagrande · Diego Andrés Cascante Camacho

**Qué hace.** Define y corre el muestreador de M1 (4 cadenas, 5.000 de calentamiento y 20.000 guardadas), calcula R̂ y ESS con ArviZ y guarda los resúmenes que usan 07 y 08.

**Entradas.** `datos/siniestros_limpios.csv`.

**Salidas.** `salidas/resultados/diagnosticos_M1.csv`, `05_M1.json`, `m1_resumen.npz`.

*Respaldo por etapas: el notebook principal es `../Grupo1_Bayesiana_completo.ipynb`, que reúne estas ocho etapas.* Orden de ejecución y trazabilidad completa: `../README.md`.

In [1]:
# Configuración: funciona en Google Colab y en un computador
# En Colab: suba la carpeta "notebooks" a su Drive. Si el buscador automático no la encuentra,
# escriba su ruta en BASE_COLAB, p. ej. '/content/drive/MyDrive/Grupo1_EntregaFinal_Bayesiana/notebooks'
BASE_COLAB = ''
import os, sys, glob
from pathlib import Path
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not BASE_COLAB:
        cand = glob.glob('/content/drive/MyDrive/**/notebooks/comun.py', recursive=True)
        assert cand, 'No encontré la carpeta notebooks en su Drive: escriba su ruta en BASE_COLAB'
        BASE_COLAB = os.path.dirname(cand[0])
    BASE = Path(BASE_COLAB)
else:
    # el notebook puede estar en notebooks/ o en notebooks/por_etapas/
    BASE = Path.cwd() if (Path.cwd() / 'comun.py').exists() else Path.cwd().parent
os.chdir(BASE)
sys.path.insert(0, str(BASE))
import numpy, pandas, scipy, matplotlib
print('Carpeta:', BASE)
print('Python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| scipy', scipy.__version__, '| matplotlib', matplotlib.__version__)

Carpeta: C:\Users\dddan\Downloads\Pensamiento - Bayesiana\4. Estadística Bayesiana\Proyecto\4. Entrega final\notebooks
Python 3.12.10 | numpy 2.3.5 | pandas 2.3.3 | scipy 1.18.1 | matplotlib 3.10.7


In [2]:
from comun import *
import time
import arviz as az
print('arviz', az.__version__)

arviz 0.23.4


## El algoritmo
Gibbs exacto para cada $\theta_c$ y paseo aleatorio de Metropolis-Hastings en $(\operatorname{logit} m, \log\kappa)$ con jacobiano. Este código es idéntico al de `mcmc.py` y al del script final.

In [3]:
# ---------------------------------------------------------------------------
# 3. Utilidades
# ---------------------------------------------------------------------------
def adaptar(logs, acc, it):
    """Ajusta la escala de la propuesta cada 100 iteraciones del calentamiento (objetivo ~0,3-0,44)."""
    if it < BURN and it > 0 and it % 100 == 0:
        tasa = acc / 100.0
        logs += np.where(tasa > 0.40, 0.15, np.where(tasa < 0.25, -0.15, 0.0))
        acc[...] = 0
    return logs


def resumen_az(post):
    idata = az.from_dict(posterior=post)
    s = az.summary(idata, kind="diagnostics")
    return s

# ---------------------------------------------------------------------------
# 4. M1: Beta-Binomial jerárquico (Gibbs + Metropolis-Hastings)
#   y_c | theta_c ~ Bin(n_c, theta_c)
#   theta_c | m, k ~ Beta(m k, (1-m) k)
#   m ~ Beta(2, 4),  k ~ Gamma(2, tasa)   (tasa 0,04 -> media 50)
# ---------------------------------------------------------------------------
def m1_logpost_hiper(u, v, theta, a_m=2.0, b_m=4.0, sh_k=2.0, rt_k=0.04):
    m, k = sp.expit(u), np.exp(v)
    al, be = m * k, (1 - m) * k
    lt, l1t = np.log(theta), np.log1p(-theta)
    ll = ((al - 1)[:, None] * lt + (be - 1)[:, None] * l1t).sum(1) - theta.shape[1] * sp.betaln(al, be)
    lp = (a_m - 1) * np.log(m) + (b_m - 1) * np.log1p(-m) + (sh_k - 1) * v - rt_k * k
    jac = np.log(m) + np.log1p(-m) + v
    return ll + lp + jac


def correr_m1(n, y, rt_k=0.04, a_m=2.0, b_m=4.0, seed=SEED, guardar_theta=True):
    rng = np.random.default_rng(seed)
    C = len(n)
    u = rng.normal(sp.logit(0.33), 0.3, K)
    v = np.log(rng.uniform(20, 100, K))
    logs = np.log(np.array([0.05, 0.15]))
    acc = np.zeros(2)
    out = {"m": np.empty((K, ITER)), "kappa": np.empty((K, ITER))}
    if guardar_theta:
        out["theta"] = np.empty((K, ITER, C), dtype=np.float32)
    nacc = 0
    for it in range(BURN + ITER):
        m, k = sp.expit(u), np.exp(v)
        # Paso de Gibbs: theta_c | m, k, y ~ Beta(m k + y_c, (1-m) k + n_c - y_c)
        theta = rng.beta((m * k)[:, None] + y, ((1 - m) * k)[:, None] + n - y)
        theta = np.clip(theta, 1e-12, 1 - 1e-12)
        # Paso de MH: (logit m, log k) con paseo aleatorio normal
        up = u + np.exp(logs[0]) * rng.standard_normal(K)
        vp = v + np.exp(logs[1]) * rng.standard_normal(K)
        lr = (m1_logpost_hiper(up, vp, theta, a_m, b_m, 2.0, rt_k)
              - m1_logpost_hiper(u, v, theta, a_m, b_m, 2.0, rt_k))
        ok = np.log(rng.uniform(size=K)) < lr
        u, v = np.where(ok, up, u), np.where(ok, vp, v)
        acc += ok.mean()
        logs = adaptar(logs, acc, it)
        if it >= BURN:
            j = it - BURN
            nacc += ok.sum()
            out["m"][:, j], out["kappa"][:, j] = sp.expit(u), np.exp(v)
            if guardar_theta:
                out["theta"][:, j] = theta
    out["acept"] = nacc / (K * ITER)
    return out

## Ajuste y diagnóstico

In [4]:
t0 = time.time()
m1 = correr_m1(n_tr, y_tr)
print(f'Tiempo: {time.time() - t0:.1f} s | aceptación MH: {m1["acept"]:.3f}')
diag1 = resumen_az({'m': m1['m'], 'kappa': m1['kappa'], 'theta': m1['theta'].astype(float)})
diag1.to_csv(RES / 'diagnosticos_M1.csv')
print('R-hat máx:', diag1.r_hat.max(), '| ESS bulk mín:', diag1.ess_bulk.min(), '| ESS tail mín:', diag1.ess_tail.min())
diag1.loc[['m', 'kappa']]

Tiempo: 2.2 s | aceptación MH: 0.320


R-hat máx: 1.0 | ESS bulk mín: 8961.0 | ESS tail mín: 12835.0


,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
m,0.00,0.000,9942.0,13750.0,1.0
kappa,0.04,0.025,8961.0,12835.0,1.0


In [5]:
th1 = m1['theta'].reshape(-1, L * F).astype(float)
i_usme = int(np.where((c_tr.localidad == 'Usme') & (c_tr.franja == 'Noche (18-24)'))[0][0])
np.savez_compressed(SALIDAS / 'm1_resumen.npz', m=m1['m'], kappa=m1['kappa'], theta_usme=m1['theta'][:, :, i_usme],
                    media=th1.mean(0), q05=np.quantile(th1, 0.05, 0), q95=np.quantile(th1, 0.95, 0),
                    p_top10=prob_top10(th1), thin20=th1[::20].astype(np.float32))
R = dict(acept=m1['acept'], m=m1['m'].mean(), m_ci=np.quantile(m1['m'], [0.05, 0.95]).tolist(),
         kappa=m1['kappa'].mean(), kappa_ci=np.quantile(m1['kappa'], [0.05, 0.95]).tolist(),
         rhat_max=diag1.r_hat.max(), ess_bulk_min=diag1.ess_bulk.min(), ess_tail_min=diag1.ess_tail.min())
print(limpio(R))
guardar_json(R, '05_M1.json')

{'acept': 0.32022, 'm': 0.3673, 'm_ci': [0.34862, 0.38623], 'kappa': 23.46104, 'kappa_ci': [17.66914, 30.12569], 'rhat_max': 1.0, 'ess_bulk_min': 8961.0, 'ess_tail_min': 12835.0}
Guardado: C:\Users\dddan\Downloads\Pensamiento - Bayesiana\4. Estadística Bayesiana\Proyecto\4. Entrega final\notebooks\salidas\resultados\05_M1.json
